In [1]:
# Function 5 - BBO Capstone Project
# Strategy: Gaussian Process + EI/UCB Hybrid
# Function 5 is a 4D black-box optimization problem.

import numpy as np
from scipy.stats import norm, qmc
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel


# --------------------------------------------------
# 1. Function 5 data
# --------------------------------------------------

X = np.array([
    [0.19144708, 0.03819337, 0.60741781, 0.41458414],
    [0.75865295, 0.53651774, 0.65600038, 0.36034155],
    [0.43834987, 0.80433970, 0.21024527, 0.15129482],
    [0.70605083, 0.53419196, 0.26424335, 0.48208755],
    [0.83647799, 0.19360965, 0.66389270, 0.78564888],
    [0.68343225, 0.11866264, 0.82904591, 0.56757661],
    [0.55362148, 0.66734998, 0.32380582, 0.81486975],
    [0.35235627, 0.32224153, 0.11697937, 0.47311252],
    [0.15378571, 0.72938169, 0.42259844, 0.44307417],
    [0.46344227, 0.63002451, 0.10790646, 0.95764390],
    [0.67749115, 0.35850951, 0.47959222, 0.07288048],
    [0.58397341, 0.14724265, 0.34809746, 0.42861465],
    [0.30688872, 0.31687813, 0.62263448, 0.09539906],
    [0.51114177, 0.81795700, 0.72871042, 0.11235362],
    [0.43893338, 0.77409176, 0.37816709, 0.93369621],
    [0.22418902, 0.84648049, 0.87948418, 0.87851568],
    [0.72526172, 0.47987049, 0.08894684, 0.75976022],
    [0.35548161, 0.63961937, 0.41761768, 0.12260384],
    [0.11987923, 0.86254031, 0.64333133, 0.84980383],
    [0.12688467, 0.15342962, 0.77016219, 0.19051811],

    # Iteration results from our project
    [0.22418900, 0.84648000, 0.87948400, 0.87851500],
    [0.86305200, 0.42515600, 0.72547900, 0.80188900],
    [0.22418902, 0.84648049, 0.87948418, 0.87851568],
    [0.38514400, 0.83522200, 0.87549200, 0.95676400],
    [0.26790600, 0.46552400, 0.22134600, 0.49407200],
    [0.60921700, 0.04459900, 0.29316900, 0.51496000],
    [0.73567600, 0.10509400, 0.84997600, 0.95476500],
    [0.57373300, 0.73742100, 0.87814500, 0.95037100],
    [0.44875300, 0.22279700, 0.87895900, 0.94933700],
    [0.46782600, 0.85464900, 0.80483500, 0.94110600],
    [0.14916300, 0.70413600, 0.87313900, 0.95036300],
    [0.34779700, 0.83339900, 0.86110600, 0.92156600],
])

y = np.array([
    64.4434399,
    18.3013796,
    0.112939795,
    4.21089813,
    258.370525,
    78.4343889,
    57.5715369,
    109.571876,
    8.84799176,
    233.223610,
    24.4230883,
    64.4201468,
    63.4767158,
    79.7291299,
    355.806818,
    1088.85962,
    28.8667516,
    45.1815703,
    431.612757,
    9.97233189,

    # Iteration outputs from our project
    1088.85351147375,
    472.699581504888,
    1088.85351147375,
    1513.30490292936,
    88.61613670263588,
    44.56790661878319,
    976.8066045249683,
    1285.4697786864638,
    677.8614123107656,
    1228.9334442204602,
    971.9245547045626,
    1194.0392012268128,
])


# --------------------------------------------------
# 2. Helper functions
# --------------------------------------------------

def format_query(point, digits=6):
    """Format a point for BBO portal submission."""
    return "-".join(f"{value:.{digits}f}" for value in point)


def clip_to_bounds(points):
    """Keep all values inside the portal domain [0, 1)."""
    return np.clip(points, 0.0, 0.999999)


def expected_improvement(candidates, model, y_best, xi=0.01):
    """Expected Improvement acquisition function."""
    mean, std = model.predict(candidates, return_std=True)
    std = np.maximum(std, 1e-12)

    improvement = mean - y_best - xi
    z = improvement / std

    ei = improvement * norm.cdf(z) + std * norm.pdf(z)
    return ei


def upper_confidence_bound(candidates, model, kappa=2.0):
    """Upper Confidence Bound acquisition function."""
    mean, std = model.predict(candidates, return_std=True)
    return mean + kappa * std


# --------------------------------------------------
# 3. Check data and current best result
# --------------------------------------------------

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

best_index = np.argmax(y)
best_point = X[best_index]
best_y = y[best_index]

print("\nBest point so far:")
print(format_query(best_point))
print("Best output so far:", best_y)


# --------------------------------------------------
# 4. Gaussian Process model
# --------------------------------------------------
# Matern kernel is suitable for nonlinear black-box functions.
# Function 5 has large output values, so normalize_y=True is important.

kernel = (
    ConstantKernel(1.0, constant_value_bounds=(1e-3, 1e3))
    * Matern(
        length_scale=[0.2, 0.2, 0.2, 0.2],
        length_scale_bounds=(1e-3, 1e3),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(1e-10, 1e-2)
    )
)

gpr = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    alpha=1e-8,
    n_restarts_optimizer=20,
    random_state=42
)

gpr.fit(X, y)

print("\nGP Model Diagnostics:")
print("Kernel:", gpr.kernel_)
print("Training score:", round(gpr.score(X, y), 4))


# --------------------------------------------------
# 5. Generate candidate points
# --------------------------------------------------
# Strategy:
# Function 5 already has a strong promising region.
# Therefore, we use more local exploitation, but still keep global exploration.

np.random.seed(42)

# Local candidates around best observed point
local_candidates = best_point + np.random.normal(
    loc=0.0,
    scale=0.08,
    size=(6000, 4)
)

local_candidates = clip_to_bounds(local_candidates)

# Global candidates across full domain [0, 1)
sampler = qmc.LatinHypercube(d=4, seed=42)
global_candidates = sampler.random(n=4000)

# Combine candidates
X_candidates = np.vstack([
    local_candidates,
    global_candidates
])


# --------------------------------------------------
# 6. Acquisition scoring: EI + UCB
# --------------------------------------------------

EI_XI = 0.01
UCB_KAPPA = 2.0

ei_scores = expected_improvement(
    candidates=X_candidates,
    model=gpr,
    y_best=best_y,
    xi=EI_XI
)

ucb_scores = upper_confidence_bound(
    candidates=X_candidates,
    model=gpr,
    kappa=UCB_KAPPA
)

# Normalize scores before combining
ei_norm = (ei_scores - ei_scores.min()) / (ei_scores.max() - ei_scores.min() + 1e-12)
ucb_norm = (ucb_scores - ucb_scores.min()) / (ucb_scores.max() - ucb_scores.min() + 1e-12)

# Hybrid score:
# 70% EI = exploitation
# 30% UCB = exploration
hybrid_score = 0.70 * ei_norm + 0.30 * ucb_norm


# --------------------------------------------------
# 7. Select next query
# --------------------------------------------------

best_candidate_index = np.argmax(hybrid_score)
x_next = X_candidates[best_candidate_index]

predicted_y, predicted_std = gpr.predict(
    x_next.reshape(1, -1),
    return_std=True
)

print("\nNext Point to Sample:")
print("X =", x_next)
print("Submission format:", format_query(x_next))
print("Predicted y:", predicted_y[0])
print("Uncertainty:", predicted_std[0])
print("Hybrid score:", hybrid_score[best_candidate_index])


# --------------------------------------------------
# 8. Show top 5 candidate points
# --------------------------------------------------

top5_index = np.argsort(hybrid_score)[-5:][::-1]

print("\nTop 5 Candidates:")

for rank, idx in enumerate(top5_index, start=1):
    point = X_candidates[idx]
    mean_i, std_i = gpr.predict(point.reshape(1, -1), return_std=True)

    print(
        f"{rank}. X={format_query(point)} | "
        f"pred={mean_i[0]:.6f}, "
        f"std={std_i[0]:.6f}, "
        f"score={hybrid_score[idx]:.6f}"
    )

Shape of X: (32, 4)
Shape of y: (32,)

Best point so far:
0.385144-0.835222-0.875492-0.956764
Best output so far: 1513.30490292936

GP Model Diagnostics:
Kernel: 0.867**2 * Matern(length_scale=[0.482, 0.791, 0.527, 0.131], nu=2.5) + WhiteKernel(noise_level=1e-10)
Training score: 1.0

Next Point to Sample:
X = [0.42927842 0.9869731  0.999999   0.99204687]
Submission format: 0.429278-0.986973-0.999999-0.992047
Predicted y: 1811.1986240141152
Uncertainty: 158.18284497733745
Hybrid score: 0.9980127719671571

Top 5 Candidates:
1. X=0.429278-0.986973-0.999999-0.992047 | pred=1811.198624, std=158.182845, score=0.998013
2. X=0.413462-0.986634-0.989874-0.999999 | pred=1806.792675, std=165.729134, score=0.991002
3. X=0.390432-0.969466-0.999999-0.999251 | pred=1805.029996, std=162.485514, score=0.985186
4. X=0.419226-0.992876-0.977009-0.999999 | pred=1804.272757, std=163.399797, score=0.983815
5. X=0.415571-0.958032-0.980308-0.999999 | pred=1805.174022, std=153.802877, score=0.981456


/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
